# Days 10-15 — CV Deepfake Classifier (EfficientNet-B0 on FaceForensics++)

**Run this in Google Colab, not locally.** `CLAUDE.md`'s hard constraint for this
project is CPU-only inference everywhere *except* model training, which happens in
Colab's free T4 GPU tier — this notebook is that training step. The resulting
checkpoint (`cv_watchdog_v1.pt`) is downloaded back into `models/` for CPU-only
local inference, same pattern as `train_nlp_classifier.py`'s DistilBERT checkpoint.

**What this notebook does, in order:**
1. Copies `ff_subset.zip` from Drive to local Colab disk and unzips it (never reads
   training data off mounted Drive — far too slow).
2. Parses FaceForensics++'s reciprocal-pair filenames into **identity groups** and
   splits train/test by group, not by file — see the dedicated section below on why
   this matters, because it's a real methodology point, not just an implementation
   detail.
3. Extracts ~10 evenly-spaced frames per video, MTCNN-crops each face
   (facenet-pytorch, mirroring `watchdog/cv_engine.py`'s detection logic so training
   and inference-time preprocessing match).
4. Fine-tunes `torchvision`'s pretrained EfficientNet-B0 for binary real/fake.
5. Evaluates on the held-out FF++ split — **with an explicit accuracy gate**. If
   this doesn't clear ~90%, the notebook stops rather than silently saving a broken
   checkpoint (see the "THE GATE" cell near the end).
6. Saves the checkpoint + its FF++ metrics to Drive for download into `models/`.

**Scope note:** `Celeb-DF-v2.zip` is staged in Drive but **not used here**. FF++
alone is Objective 3's first pass; Celeb-DF is reserved as a second, independent
evaluation domain if the FF++-trained model looks credible — same "prove it works
on one clean benchmark before adding a second" discipline as the NLP classifier's
build order. Using it now would just multiply the runtime and the identity-leakage
bookkeeping below without adding information yet.

**What happens after this notebook:** download `cv_watchdog_v1.pt` and
`cv_watchdog_v1_ff_metrics.json` from Drive into this project's local `models/`
directory, then run `python evaluate_cv_transfer.py` locally — that script loads
this checkpoint and measures it **zero-shot** against this project's own 32
CV-eligible Nigerian clips. The drop between this notebook's FF++ number and that
script's Nigerian-corpus number is the actual result: the CV counterpart to the NLP
engine's Day 3 "perplexity runs backwards on Nigerian-register text" finding,
testing §2.3.4's claim that Western-trained detectors degrade on Global South
media. A model that clears the gate here and then collapses on the Nigerian corpus
is not a failed notebook — that gap **is** the finding.


## 0. Runtime check

**Before running anything below:** `Runtime` menu → `Change runtime type` → Hardware
accelerator → **T4 GPU**, then run this cell. If it doesn't report a GPU, fix the
runtime type first — training EfficientNet-B0 on CPU here would defeat the entire
point of using Colab (see `CLAUDE.md`'s hard constraint: training happens in Colab
*because* CPU can't handle it, everything else in this project is CPU-only).


In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected. Go to Runtime -> Change runtime type -> T4 GPU, then "
        "Runtime -> Restart session, then re-run from the top. Training on CPU here "
        "defeats the reason this notebook exists (see the intro cell)."
    )

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"torch: {torch.__version__}")


## 1. Install the one dependency Colab doesn't already have

`torch`/`torchvision`/`pandas`/`numpy`/`scikit-learn`/`Pillow`/`tqdm`/`opencv-python`
all ship preinstalled in the Colab runtime. `facenet-pytorch` (the MTCNN
implementation `watchdog/cv_engine.py` uses locally — see
`benchmark_face_detection.py` for why it was chosen over the TensorFlow-based
`mtcnn` package) is the only thing missing.


In [ ]:
!pip install -q facenet-pytorch==2.6.0


In [ ]:
import json
import re
import time
import zipfile
from collections import defaultdict
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.model_selection import StratifiedGroupKFold
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device("cuda")
print(f"Using device: {DEVICE}")


## 2. Mount Drive, stage data on local disk

Copies the zip from Drive to `/content/` (Colab's local SSD) and unzips there.
Training reads frames extracted from `/content/ff_subset/`, never from the mounted
Drive path — Drive is a network mount from Colab's perspective and is far too slow
for the random small-file access pattern of frame extraction across ~500 videos.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Adjust this if your Drive layout differs.
DRIVE_ZIP_PATH = Path("/content/drive/MyDrive/ff_subset.zip")
LOCAL_ZIP_PATH = Path("/content/ff_subset.zip")
FFPP_ROOT = Path("/content/ff_subset")

assert DRIVE_ZIP_PATH.exists(), f"Not found: {DRIVE_ZIP_PATH} -- check the path/filename in your Drive."

t0 = time.perf_counter()
if not LOCAL_ZIP_PATH.exists():
    print(f"Copying {DRIVE_ZIP_PATH} -> {LOCAL_ZIP_PATH} ...")
    import shutil
    shutil.copy(DRIVE_ZIP_PATH, LOCAL_ZIP_PATH)
print(f"Copied in {time.perf_counter() - t0:.1f}s ({LOCAL_ZIP_PATH.stat().st_size / 1e6:.0f} MB)")

if not FFPP_ROOT.exists():
    t0 = time.perf_counter()
    print(f"Unzipping to {FFPP_ROOT} ...")
    with zipfile.ZipFile(LOCAL_ZIP_PATH) as zf:
        zf.extractall(FFPP_ROOT)
    print(f"Unzipped in {time.perf_counter() - t0:.1f}s")
else:
    print(f"{FFPP_ROOT} already exists, skipping unzip (delete it to force a fresh unzip).")

# Some FF++ zip exports wrap everything in a top-level "ff_data" folder, some
# don't -- rather than hardcoding either assumption into the discovery globs in
# the next section, just point FFPP_ROOT at wherever original_sequences/ and
# manipulated_sequences/ actually ended up. Works whether or not the wrapper
# folder exists, so a future re-export with a different layout doesn't need a
# code change here.
if (FFPP_ROOT / "ff_data").exists():
    FFPP_ROOT = FFPP_ROOT / "ff_data"
    print(f"Found an ff_data/ wrapper folder inside the zip -- using {FFPP_ROOT} as the data root.")


## 3. Identity-grouped train/test split — read this before touching the split logic

**Why splitting by file leaks identities.** FaceForensics++'s manipulated videos are
named as reciprocal pairs: `654_648.mp4` and `648_654.mp4` are the same two
identities face-swapped onto each other, both derived from originals `654.mp4` and
`648.mp4`. If `654_648.mp4` lands in train and `648_654.mp4` lands in test (or either
original lands opposite its own manipulated pair), identity 648's actual face
appears on both sides of the split. A model can then partly learn "this specific
person's face" as a shortcut for real/fake, rather than a genuine, generalisable
manipulation artifact — the resulting test accuracy would be measuring memorisation,
not detection. This is the same principle `train_nlp_classifier.py` applies via
`StratifiedGroupKFold` grouped by publisher/generation-batch, applied here to face
identity instead.

**Why grouping by *pair* isn't enough on its own.** Deepfakes and NeuralTextures in
this subset are applied to **overlapping identity pools** — the same original
identity can appear in a Deepfakes pair AND a NeuralTextures pair (possibly with a
different partner in each). So the correct group isn't "the two IDs in this one
filename" — it's every identity number **transitively connected** through any
manipulated-video filename, across both methods together. Two identities that never
appear in the same filename can still need to be in the same group if a third
identity links them (e.g. `033_071` and `071_099` put 033, 071, and 099 all in one
group, even though 033 and 099 never appear together directly). The code below
builds this via union-find over every manipulated filename's two identity numbers,
**combining Deepfakes and NeuralTextures into one graph**, not two separate ones.

**Both of these — the pair structure, and the identity overlap between methods —
belong in the write-up's methodology section (Chapter 3), not just this comment.**
A reader evaluating whether the split is sound needs to know both facts to judge it.

**What actually enforces this below:** every video (original or manipulated) is
assigned to a `group_id` (its identity's connected-component root). The train/test
split is then performed on GROUPS via `StratifiedGroupKFold`, never on individual
files — followed by an explicit assertion that no raw identity number appears in
both splits, so a bug in the grouping logic itself would fail loudly here rather
than silently degrading the FF++ number this notebook's whole gate depends on.


In [ ]:
# FaceForensics++'s standard released layout (adjust the globs below if your
# ff_subset.zip differs -- the discovery step immediately after prints what it
# actually found, specifically so a layout mismatch is caught here, not silently).
ORIGINAL_GLOB = "original_sequences/**/c23/videos/*.mp4"
DEEPFAKES_GLOB = "manipulated_sequences/Deepfakes/c23/videos/*.mp4"
NEURALTEXTURES_GLOB = "manipulated_sequences/NeuralTextures/c23/videos/*.mp4"

originals = sorted(FFPP_ROOT.glob(ORIGINAL_GLOB))
deepfakes = sorted(FFPP_ROOT.glob(DEEPFAKES_GLOB))
neuraltextures = sorted(FFPP_ROOT.glob(NEURALTEXTURES_GLOB))

print(f"originals:       {len(originals)}  (expected ~200)")
print(f"Deepfakes:       {len(deepfakes)}  (expected ~200)")
print(f"NeuralTextures:  {len(neuraltextures)}  (expected ~100)")

for name, found, expected in [("originals", len(originals), 200),
                               ("Deepfakes", len(deepfakes), 200),
                               ("NeuralTextures", len(neuraltextures), 100)]:
    if abs(found - expected) > 0.2 * expected:
        print(f"  ** WARNING: {name} count ({found}) is far from the expected ~{expected} -- "
              f"check ORIGINAL_GLOB/DEEPFAKES_GLOB/NEURALTEXTURES_GLOB above against your "
              f"actual zip layout before trusting anything downstream. **")

assert originals and deepfakes, "Found no videos at all -- the glob patterns almost certainly don't match this zip's layout."


In [ ]:
# ---- identity parsing + union-find over BOTH manipulation methods together ----
PAIR_RE = re.compile(r"^(\d+)_(\d+)\.mp4$")
ORIG_RE = re.compile(r"^(\d+)\.mp4$")


class UnionFind:
    """Minimal union-find over arbitrary hashable ids (here: FF++ identity numbers)."""
    def __init__(self):
        self.parent = {}

    def find(self, x):
        self.parent.setdefault(x, x)
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb:
            self.parent[ra] = rb


uf = UnionFind()
for path in deepfakes + neuraltextures:
    m = PAIR_RE.match(path.name)
    assert m, f"Filename doesn't match the expected id_id.mp4 pattern: {path.name}"
    a, b = int(m.group(1)), int(m.group(2))
    uf.union(a, b)
for path in originals:
    m = ORIG_RE.match(path.name)
    assert m, f"Filename doesn't match the expected id.mp4 pattern: {path.name}"
    uf.find(int(m.group(1)))  # register even if it never appears in a manipulated pair

def video_record(path, label, method):
    m = PAIR_RE.match(path.name)
    if m:
        a, b = int(m.group(1)), int(m.group(2))
        ids = {a, b}
    else:
        a = int(ORIG_RE.match(path.name).group(1))
        ids = {a}
    return {"path": str(path), "label": label, "method": method,
            "group_id": uf.find(next(iter(ids))), "identity_ids": ids}

video_rows = (
    [video_record(p, 0, "ORIGINAL") for p in originals]
    + [video_record(p, 1, "Deepfakes") for p in deepfakes]
    + [video_record(p, 1, "NeuralTextures") for p in neuraltextures]
)
videos_df = pd.DataFrame(video_rows)
print(f"{len(videos_df)} videos total, {videos_df['group_id'].nunique()} identity groups")
print(videos_df.groupby(["method", "label"]).size())


In [ ]:
# ---- group-based, label-stratified split (one held-out fold, same tool as
#      train_nlp_classifier.py's StratifiedGroupKFold, used here for a single
#      80/20-ish split rather than full k-fold CV -- Colab session time is
#      limited and one clean held-out number is what the gate below needs) ----
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
train_idx, test_idx = next(sgkf.split(
    X=videos_df.index, y=videos_df["label"], groups=videos_df["group_id"],
))
train_videos = videos_df.iloc[train_idx].reset_index(drop=True)
test_videos = videos_df.iloc[test_idx].reset_index(drop=True)

print(f"train: {len(train_videos)} videos  {train_videos['label'].value_counts().to_dict()}")
print(f"test:  {len(test_videos)} videos  {test_videos['label'].value_counts().to_dict()}")

# ---- THE assertion this whole section exists for ----
train_identities = set().union(*train_videos["identity_ids"]) if len(train_videos) else set()
test_identities = set().union(*test_videos["identity_ids"]) if len(test_videos) else set()
leaked = train_identities & test_identities
assert not leaked, f"IDENTITY LEAK across train/test: {leaked}"
print(f"OK -- {len(train_identities)} train identities / {len(test_identities)} test identities, zero overlap.")


In [ ]:
# ---- internal validation split, carved out of TRAIN only ----
# The 12-point train/test gap the previous run produced (train_acc 0.978 vs
# held-out video accuracy 0.86) was diagnosed with zero visibility into *when*
# during training the model started overfitting -- section 6 previously only
# ever logged training accuracy. The FF++ test set above is (correctly) only
# touched once, in section 7, specifically so it stays a genuine held-out
# number for the section 8 gate -- so rather than peek at it more than once,
# carve a second, disjoint group-based split out of TRAIN itself, exactly the
# same way (same union-find groups, same StratifiedGroupKFold tool, same leak
# assertion) as the train/test split above, just one level in. This gives
# per-epoch validation metrics during training without the test set ever being
# touched before section 7.
val_sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
fit_idx, val_idx = next(val_sgkf.split(
    X=train_videos.index, y=train_videos["label"], groups=train_videos["group_id"],
))
fit_videos = train_videos.iloc[fit_idx].reset_index(drop=True)
val_videos = train_videos.iloc[val_idx].reset_index(drop=True)

print(f"fit: {len(fit_videos)} videos  {fit_videos['label'].value_counts().to_dict()}")
print(f"val: {len(val_videos)} videos  {val_videos['label'].value_counts().to_dict()}")

fit_identities = set().union(*fit_videos["identity_ids"]) if len(fit_videos) else set()
val_identities = set().union(*val_videos["identity_ids"]) if len(val_videos) else set()
leaked_fit_val = fit_identities & val_identities
assert not leaked_fit_val, f"IDENTITY LEAK across fit/val: {leaked_fit_val}"
print(f"OK -- {len(fit_identities)} fit identities / {len(val_identities)} val identities, zero overlap.")
print("(val_videos is a disjoint subset of train_videos, so its crops come from the same "
      "extraction/caching pass as the rest of train/ below -- no separate extraction step "
      "or Drive cache entry needed for it.)")


## 4. Frame extraction + MTCNN face crop

The crop function below (`letterbox()` + `detect_best_face_bbox()`) is a **manual
duplicate** of the relevant parts of `watchdog/cv_engine.py`'s `letterbox()` /
`detect_face_bbox()` / `_load_winning_mtcnn()` — same constants
(`MTCNN_MIN_FACE_SIZE=40`, `keep_all=True`, a 640x640 letterboxed detector input),
same "never stretch, pad to square" reasoning. It's duplicated rather than imported
because this notebook runs in Colab against a standalone `facenet-pytorch` install,
not this project's full local package (which also pulls in `mediapipe`, `psycopg2`,
`sentence-transformers`, etc. — unnecessary weight for a one-off training run, and
a much slower Colab setup cell for no benefit here). **This is a real maintenance
risk, stated plainly rather than hidden**: if `cv_engine.py`'s detection logic
changes, this copy will silently drift out of sync unless someone updates both.
There is no tracking, and the two files have already fully diverged. There's one
deliberate simplification versus the original: `detect_face_bbox()` tracks
continuity against a previous frame's box for temporal video analysis; here every
sampled frame is an independent training example with no previous box to track
against, so this only ever takes the single highest-confidence detection per frame
(exactly `detect_face_bbox()`'s own fallback path when `prev_box=None`).

**One more asymmetry, also disclosed rather than left implicit: this notebook's
MTCNN runs on `device=DEVICE` (CUDA)** — it's processing ~500 training videos, and
GPU is the entire reason this step runs in Colab — **while `cv_engine._load_winning_mtcnn()`
(the detector `evaluate_cv_transfer.py` actually calls for the Nigerian-corpus side
of this comparison) hardcodes `device="cpu"`**, per `CLAUDE.md`'s CPU-only-outside-Colab
constraint. This is a necessary consequence of that constraint, not an oversight —
and every other piece of the two pipelines (letterbox, crop margin, frame-sampling
formula, box-selection logic, output normalization) was checked and matches
exactly. But `facenet-pytorch`'s GPU and CPU code paths aren't guaranteed
bit-identical: floating-point non-associativity through PNet/RNet/ONet means a
face sitting right at the detector's confidence threshold could in principle be
detected on one device and missed on the other. Low-probability in practice, but
it's the one non-identical piece in an otherwise matched preprocessing pipeline
being used to measure a domain-transfer gap, so it belongs in Chapter 3/4 as an
explicit caveat alongside the FF++-vs-Nigerian result (see also
`docs/PROJECT_CONTEXT.md`) — not as an implicit assumption of parity.


In [ ]:
from facenet_pytorch import MTCNN as FacenetMTCNN

MTCNN_MIN_FACE_SIZE = 40          # must match config.MTCNN_MIN_FACE_SIZE
LETTERBOX_TARGET_SIZE = (640, 640)  # must match config.LETTERBOX_TARGET_SIZE
CROP_MARGIN_FRAC = 0.3             # must match cv_engine._CROP_MARGIN_FRAC
CROP_OUTPUT_SIZE = (224, 224)      # EfficientNet-B0's expected input resolution

_mtcnn_detector = FacenetMTCNN(keep_all=True, device=DEVICE, min_face_size=MTCNN_MIN_FACE_SIZE)


def letterbox(frame: np.ndarray, target_size=LETTERBOX_TARGET_SIZE):
    """Verbatim port of watchdog/cv_engine.py's letterbox() -- resize preserving
    aspect ratio, pad with black to target_size. Never stretches (facial geometry
    distortion would be exactly the kind of artifact this pipeline hunts for)."""
    h, w = frame.shape[:2]
    target_w, target_h = target_size
    scale = min(target_w / w, target_h / h)
    new_w, new_h = int(round(w * scale)), int(round(h * scale))
    resized = cv2.resize(frame, (new_w, new_h), interpolation=cv2.INTER_AREA)
    canvas = np.zeros((target_h, target_w, 3), dtype=frame.dtype)
    pad_x, pad_y = (target_w - new_w) // 2, (target_h - new_h) // 2
    canvas[pad_y:pad_y + new_h, pad_x:pad_x + new_w] = resized
    return canvas, scale, (pad_x, pad_y)


def unletterbox_box(box, scale, pad):
    x, y, w, h = box
    pad_x, pad_y = pad
    return ((x - pad_x) / scale, (y - pad_y) / scale, w / scale, h / scale)


def detect_best_face_bbox(frame_bgr: np.ndarray):
    """Single highest-confidence MTCNN detection in ORIGINAL frame coordinates,
    or None. See the markdown cell above for how this relates to cv_engine.detect_face_bbox()."""
    letterboxed, scale, pad = letterbox(frame_bgr)
    rgb = cv2.cvtColor(letterboxed, cv2.COLOR_BGR2RGB)
    boxes, probs = _mtcnn_detector.detect(rgb)
    if boxes is None:
        return None
    candidates = [(unletterbox_box((x1, y1, x2 - x1, y2 - y1), scale, pad), float(p))
                  for (x1, y1, x2, y2), p in zip(boxes, probs) if p is not None]
    if not candidates:
        return None
    return max(candidates, key=lambda c: c[1])[0]


def crop_face(frame_bgr: np.ndarray, box, margin_frac=CROP_MARGIN_FRAC):
    x, y, w, h = box
    mx, my = w * margin_frac, h * margin_frac
    x1, y1 = max(0, int(x - mx)), max(0, int(y - my))
    x2, y2 = min(frame_bgr.shape[1], int(x + w + mx)), min(frame_bgr.shape[0], int(y + h + my))
    if x2 <= x1 or y2 <= y1:
        return None
    return frame_bgr[y1:y2, x1:x2]


In [ ]:
def sample_frame_indices(n_total_frames: int, n_samples: int = 10):
    if n_total_frames <= 0:
        return []
    n_samples = min(n_samples, n_total_frames)
    return sorted(set(int(i) for i in np.linspace(0, n_total_frames - 1, n_samples)))


def extract_face_crops(video_path: str, n_samples: int = 10):
    """Returns a list of BGR face-crop arrays (letterboxed to CROP_OUTPUT_SIZE),
    one per sampled frame that actually had a detectable face. Frames with no
    detected face are skipped, not padded with a blank crop -- a fabricated crop
    would be exactly the kind of silent-corruption bug this project's bug log
    (docs/PROJECT_CONTEXT.md #6/#9/#10) warns about."""
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return []
    n_total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    crops = []
    for idx in sample_frame_indices(n_total, n_samples):
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ok, frame = cap.read()
        if not ok:
            continue
        box = detect_best_face_bbox(frame)
        if box is None:
            continue
        crop = crop_face(frame, box)
        if crop is None or crop.size == 0:
            continue
        squared, _, _ = letterbox(crop, CROP_OUTPUT_SIZE)
        crops.append(squared)
    cap.release()
    return crops


### Resume-safe extraction

Frame extraction below is the slowest, most I/O-bound step in this notebook (see
the runtime note in the closing section) and previously only ever lived on
Colab's ephemeral `/content` disk — a disconnect anywhere downstream, including
during training, meant redoing all of it from scratch. The cell below backs the
extracted crops up to Drive (`FRAMES_ZIP_DRIVE_PATH`) right after extraction
finishes, and restores from that cache instead of re-extracting if it's already
there. Delete the Drive cache manually if you deliberately change the FF++
subset, `SEED`, or the split logic and need a genuinely fresh extraction — this
cell has no way to detect on its own that a stale cache no longer matches the
current split, beyond the empty-crops warning it prints.


In [ ]:
FRAMES_DIR = Path("/content/frames")
FRAMES_DIR.mkdir(exist_ok=True)

# Extraction is the slow, I/O-bound part of this notebook (video seeking, not
# GPU compute) and, unlike the checkpoint in section 9, it used to only ever
# live on /content, Colab's ephemeral local disk -- a T4 reclaim mid-training
# meant redoing all of it. FRAMES_ZIP_DRIVE_PATH backs the extracted crops up
# to Drive right after extraction finishes, and is checked FIRST on every run
# -- if it's already there, this cell restores from it instead of re-extracting.
FRAMES_ZIP_DRIVE_PATH = Path("/content/drive/MyDrive/watchdog_checkpoints/ff_frames_cache.zip")


def extract_and_save(videos_df: pd.DataFrame, split_name: str):
    rows = []
    zero_crop_videos = []
    out_dir = FRAMES_DIR / split_name
    out_dir.mkdir(parents=True, exist_ok=True)

    for i, row in videos_df.iterrows():
        stem = Path(row["path"]).stem + f"_{row['method']}"
        crops = extract_face_crops(row["path"])
        if not crops:
            zero_crop_videos.append(row["path"])
        for j, crop in enumerate(crops):
            crop_path = out_dir / f"{stem}_{j}.jpg"
            cv2.imwrite(str(crop_path), crop)
            rows.append({"crop_path": str(crop_path), "label": row["label"],
                         "video_path": row["path"], "method": row["method"]})
        if (i + 1) % 50 == 0:
            print(f"  [{split_name}] {i + 1}/{len(videos_df)} videos processed, {len(rows)} crops so far")

    if zero_crop_videos:
        print(f"  ** {len(zero_crop_videos)}/{len(videos_df)} {split_name} videos yielded ZERO face crops "
              f"(no detectable face in any sampled frame) -- excluded from training/eval, not padded: **")
        for p in zero_crop_videos[:10]:
            print(f"     {p}")
        if len(zero_crop_videos) > 10:
            print(f"     ... and {len(zero_crop_videos) - 10} more")

    return pd.DataFrame(rows)


def crops_df_from_disk(videos_df: pd.DataFrame, split_name: str) -> pd.DataFrame:
    """Reconstructs the DataFrame extract_and_save() would have produced, by
    matching each video's crop-filename stem against files already on disk --
    used on the restore path below so the Drive cache doesn't need its own
    separate metadata file, just the (deterministically reproducible)
    train_videos/test_videos split plus the crop .jpg files themselves."""
    out_dir = FRAMES_DIR / split_name
    rows = []
    for _, row in videos_df.iterrows():
        stem = Path(row["path"]).stem + f"_{row['method']}"
        for crop_path in sorted(out_dir.glob(f"{stem}_*.jpg")):
            rows.append({"crop_path": str(crop_path), "label": row["label"],
                         "video_path": row["path"], "method": row["method"]})
    return pd.DataFrame(rows)


t0 = time.perf_counter()
if FRAMES_ZIP_DRIVE_PATH.exists():
    print(f"Found a cached extraction at {FRAMES_ZIP_DRIVE_PATH} -- restoring instead of "
          f"re-extracting (delete it from Drive to force a fresh extraction).")
    with zipfile.ZipFile(FRAMES_ZIP_DRIVE_PATH) as zf:
        zf.extractall(FRAMES_DIR)
    train_crops = crops_df_from_disk(train_videos, "train")
    test_crops = crops_df_from_disk(test_videos, "test")
    print(f"Restored in {time.perf_counter() - t0:.1f}s")
    print(f"train crops: {len(train_crops)}  {train_crops['label'].value_counts().to_dict()}")
    print(f"test crops:  {len(test_crops)}  {test_crops['label'].value_counts().to_dict()}")
    if train_crops.empty or test_crops.empty:
        print("  ** WARNING: restored 0 crops for a split -- the Drive cache likely doesn't "
              "match the current train/test split (e.g. SEED, the FF++ subset, or the split "
              "logic changed since it was written). Delete the Drive cache and re-run this "
              "cell to force a fresh extraction rather than trusting a stale/mismatched one. **")
else:
    train_crops = extract_and_save(train_videos, "train")
    test_crops = extract_and_save(test_videos, "test")
    print(f"\nExtraction took {time.perf_counter() - t0:.1f}s")
    print(f"train crops: {len(train_crops)}  {train_crops['label'].value_counts().to_dict()}")
    print(f"test crops:  {len(test_crops)}  {test_crops['label'].value_counts().to_dict()}")

    print(f"\nBacking up extracted crops to Drive ({FRAMES_ZIP_DRIVE_PATH}) so a later "
          f"disconnect (e.g. mid-training) doesn't force redoing this step...")
    t1 = time.perf_counter()
    import shutil
    FRAMES_ZIP_DRIVE_PATH.parent.mkdir(parents=True, exist_ok=True)
    local_backup_zip = Path("/content/frames_backup.zip")
    with zipfile.ZipFile(local_backup_zip, "w", zipfile.ZIP_STORED) as zf:  # STORED: jpgs are already compressed
        for p in FRAMES_DIR.rglob("*.jpg"):
            zf.write(p, p.relative_to(FRAMES_DIR))
    shutil.copy(local_backup_zip, FRAMES_ZIP_DRIVE_PATH)
    print(f"Backed up in {time.perf_counter() - t1:.1f}s "
          f"({local_backup_zip.stat().st_size / 1e6:.0f} MB)")


In [ ]:
# fit_crops/val_crops is a partition of the crops already extracted into
# train/ above, split by which video they came from -- no new extraction or
# Drive cache entry needed, this is purely a DataFrame split.
val_video_paths = set(val_videos["path"])
fit_crops = train_crops[~train_crops["video_path"].isin(val_video_paths)].reset_index(drop=True)
val_crops = train_crops[train_crops["video_path"].isin(val_video_paths)].reset_index(drop=True)
print(f"fit crops: {len(fit_crops)}  {fit_crops['label'].value_counts().to_dict()}")
print(f"val crops: {len(val_crops)}  {val_crops['label'].value_counts().to_dict()}")


### Sanity check before spending GPU time — look at real crops

Per this project's own working-style convention ("test against real sources early,"
`docs/PROJECT_CONTEXT.md` §6): a clean exit code from the extraction cell above is
not evidence the crops are actually faces and not e.g. letterboxed black frames or
misaligned boxes. Look at a handful before training on them.


In [ ]:
import matplotlib.pyplot as plt

sample_paths = train_crops["crop_path"].sample(min(8, len(train_crops)), random_state=SEED).tolist()
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, p in zip(axes.flat, sample_paths):
    img = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    ax.set_title(Path(p).name, fontsize=7)
    ax.axis("off")
plt.tight_layout()
plt.show()
print("Confirm these are real, reasonably-aligned face crops before proceeding.")


## 5. Dataset / DataLoader

A custom `Dataset` reading `(crop_path, label)` pairs directly from the crops
DataFrame — deliberately **not** `torchvision.datasets.ImageFolder`. `ImageFolder`
assigns class indices by alphabetically sorting folder names, which would silently
map `"fake" -> 0, "real" -> 1"` — the opposite of this project's label convention
everywhere else (`watchdog/dataset_manager.py`: `LABEL_AUTHENTIC=0`,
`LABEL_SYNTHETIC=1`). Reading the label explicitly from the DataFrame avoids that
class of silent, easy-to-miss bug entirely.


In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    # Crops are tightly framed at one fixed scale/margin
    # (cv_engine._CROP_MARGIN_FRAC, ported verbatim in section 4) -- every
    # training example sees "the face" at exactly the same relative size and
    # position, which is itself a memorizable shortcut on a small,
    # low-identity-diversity dataset. RandomResizedCrop perturbs scale and
    # position; RandomRotation covers the small in-camera tilts a real capture
    # produces. Both are standard, cheap augmentations for this task, added
    # alongside the flip/colour-jitter that were already here.
    transforms.RandomResizedCrop(CROP_OUTPUT_SIZE[0], scale=(0.85, 1.0), ratio=(0.95, 1.05)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


class FaceCropDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform):
        self.paths = df["crop_path"].tolist()
        self.labels = df["label"].tolist()
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")  # crops are already 224x224 BGR->RGB via cv2.imwrite/PIL
        return self.transform(img), self.labels[idx]


BATCH_SIZE = 32

# train_loader trains on fit_crops (train minus the carved-out internal val
# videos), not the full train_crops -- val_loader below is what section 6's
# per-epoch monitoring uses to watch the train/val gap live during training.
train_loader = DataLoader(FaceCropDataset(fit_crops, train_transform),
                           batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(FaceCropDataset(val_crops, eval_transform),
                         batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(FaceCropDataset(test_crops, eval_transform),
                          batch_size=BATCH_SIZE, shuffle=False, num_workers=2)


## 6. Model — pretrained EfficientNet-B0, fine-tuned in two phases

Binary classification via a 2-unit head + cross-entropy (same pattern as
`train_nlp_classifier.py`'s DistilBERT head: `argmax` for the predicted label,
`softmax[:, 1]` for `P(synthetic)`), not a single sigmoid unit — kept consistent
with that script so both checkpoints are loaded/used the same way at inference
time. **Loss is unweighted** (plain `CrossEntropyLoss`, no class weighting) —
this model's job is to be a representative Western-trained detector so the FF++
→ Nigerian-corpus transfer gap (§2.3.4) is measurable; biasing it toward this
project's own false-positive preferences (NFR-01/NFR-03, which govern the
*deployed* pipeline, not this baseline) would make it less representative of the
thing being measured. Any specificity/recall asymmetry in the results below is
reported as an observed property of the baseline, not corrected for here.

**Revision history, both changes made before training rather than by widening
the gate:**

- **Run 1 (single-phase, 6 epochs, lr=1e-4):** train_acc=0.978 vs held-out video
  accuracy=0.86 — a 12-point gap, loss still falling at epoch 6. High and
  improving train accuracy alongside a wide test gap is regularization/
  overfitting, not underfitting; more epochs on that setup would mainly have
  deepened the memorization. `4000 crops` also overstated real diversity: ~10
  near-duplicate crops per video from ~64 train identity groups. Fixed with
  three changes: (1) an internal validation split carved out of TRAIN only
  (`fit_videos`/`val_videos` above), so overfitting is visible per epoch — the
  FF++ test set is still touched exactly once, in section 7; (2) stronger
  augmentation (`RandomResizedCrop` + `RandomRotation` added alongside the
  flip/colour-jitter that were already here, since the crops are otherwise
  framed identically every time — see section 5); (3) freeze-then-unfreeze —
  head-only warmup, then unfreeze at a much lower LR.
- **Run 2 (two-phase, 3 head-only + 9 fine-tune @ lr=1e-5):** gap closed
  (+0.12 → +0.06), but overcorrected the other way — val_video_acc was still
  climbing at epoch 9 (0.845 → 0.845 → 0.869) and train_acc only reached
  0.869 (vs run 1's 0.978). Not converged, rather than converged badly:
  underfitting, not overfitting. Fixed with one change, everything else held
  identical: **`FINETUNE_EPOCHS` raised 9 → 20, fine-tune lr raised 1e-5 →
  5e-5.** Best-checkpoint-by-val-video-accuracy selection means the extra
  epochs can't make the result worse — if it starts overfitting again at,
  say, epoch 14, epoch 13's weights are what gets kept and reported.

**This is the last scheduled attempt.** The checkpoint carried into section 7
is whichever epoch (across both phases) scored best on the internal val
split's video-level accuracy — not necessarily the last epoch. **The section 8
gate (0.90) is unchanged.** If this run still doesn't clear it, the honest move
is to report the FF++ baseline as measured and report no transfer number — a
genuine §1.7 limitation and §5.5 further-studies item about data-scale
requirements at this identity-group count, not something to fix by widening
the gate.


In [ ]:
model = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, 2)  # 0=AUTHENTIC/real, 1=SYNTHETIC/fake
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()  # unweighted -- see section 6 markdown


def set_backbone_trainable(model, trainable: bool):
    """Freezes/unfreezes everything except the classifier head, and puts the
    backbone's BatchNorm layers in eval mode while frozen so their running
    stats don't keep drifting from small-batch statistics while only the head
    is supposed to be learning."""
    for name, param in model.named_parameters():
        if not name.startswith("classifier"):
            param.requires_grad = trainable
    model.features.train(trainable)
    model.classifier.train(True)


# ---- two-phase schedule ----
# Phase 1 (head-only): fit_crops is a few thousand crops from ~64 train
# identity groups -- small for fine-tuning a full EfficientNet-B0 from step
# one. Letting randomly-initialized head gradients flow straight through the
# whole pretrained backbone on the very first batches pushes early conv/BN
# weights toward this dataset's specific quirks (lighting, c23 compression
# artifacts, background inside the crop margin) before the head has learned
# anything useful to backprop. Training the head alone first, backbone
# frozen, lets it converge on top of stable pretrained features before the
# backbone is allowed to move at all.
# Phase 2 (fine-tune): unfreeze everything, at a lower LR and higher weight
# decay than the original single-phase run (which used lr=1e-4, wd=0.01
# default) -- gentle adjustment of already-good features, not free
# re-learning. Revised a second time (see section 6 markdown): the first
# two-phase attempt (9 epochs @ lr=1e-5) undercorrected into underfitting --
# val_video_acc was still climbing at epoch 9 and train_acc only reached
# 0.869. FINETUNE_EPOCHS raised to 20 and lr raised to 5e-5 so phase 2 can
# actually reach a good fit; best-checkpoint-by-val-video-accuracy selection
# (below) means the extra epochs can't make the final checkpoint worse --
# if it starts overfitting at, say, epoch 14, epoch 13's weights are what
# gets kept.
HEAD_EPOCHS = 3
FINETUNE_EPOCHS = 20

head_optimizer = torch.optim.AdamW(
    [p for p in model.classifier.parameters() if p.requires_grad], lr=1e-3, weight_decay=0.01,
)
finetune_optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.05)

print(model.classifier)


In [ ]:
def predict_probs(loader):
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE)
            probs = torch.softmax(model(images), dim=1)[:, 1]
            all_probs.extend(probs.cpu().tolist())
            all_labels.extend(labels.tolist())
    return np.array(all_probs), np.array(all_labels)


def compute_video_level_accuracy(loader, crops_df):
    """Same video-level aggregation section 7 uses for the real held-out
    result (mean P(synthetic) across a video's crops, thresholded at 0.5) --
    reused here for per-epoch validation monitoring so checkpoint selection
    below uses the metric the section 8 gate actually cares about, not just
    crop-level accuracy. Named distinctly from section 8's own
    `video_level_accuracy` variable (a float, not a function) so the two
    don't collide if this notebook is ever re-run cell-by-cell out of order."""
    probs, _ = predict_probs(loader)
    vc = crops_df.reset_index(drop=True).copy()
    vc["prob_synthetic"] = probs
    agg = vc.groupby("video_path").agg(
        label=("label", "first"), prob_synthetic=("prob_synthetic", "mean"),
    ).reset_index()
    agg["pred"] = (agg["prob_synthetic"] >= 0.5).astype(int)
    return accuracy_score(agg["label"], agg["pred"])


In [ ]:
import copy


def run_epoch(loader, optimizer=None):
    """One pass over `loader`. Trains if `optimizer` is given, otherwise just
    evaluates (used for the per-epoch val pass below, sharing this loop so
    train/val loss and crop-accuracy are computed identically)."""
    is_train = optimizer is not None
    model.train(is_train) if is_train else model.eval()
    epoch_loss, n_correct, n_total = 0.0, 0, 0
    with torch.enable_grad() if is_train else torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            if is_train:
                optimizer.zero_grad()
            logits = model(images)
            loss = criterion(logits, labels)
            if is_train:
                loss.backward()
                optimizer.step()
            epoch_loss += loss.item() * images.size(0)
            n_correct += (logits.argmax(dim=1) == labels).sum().item()
            n_total += images.size(0)
    return epoch_loss / n_total, n_correct / n_total


def run_phase(name, epochs, optimizer, best_val_acc, best_state):
    for epoch in range(epochs):
        t0 = time.perf_counter()
        train_loss, train_acc = run_epoch(train_loader, optimizer)
        val_loss, val_crop_acc = run_epoch(val_loader, optimizer=None)
        val_vid_acc = compute_video_level_accuracy(val_loader, val_crops)
        print(f"[{name}] epoch {epoch + 1}/{epochs}  loss={train_loss:.4f}  "
              f"train_acc={train_acc:.4f}  val_crop_acc={val_crop_acc:.4f}  "
              f"val_video_acc={val_vid_acc:.4f}  gap={train_acc - val_crop_acc:+.4f}  "
              f"({time.perf_counter() - t0:.1f}s)")
        if val_vid_acc > best_val_acc:
            best_val_acc = val_vid_acc
            best_state = copy.deepcopy(model.state_dict())
    return best_val_acc, best_state


best_val_acc, best_state = -1.0, None

print(f"-- Phase 1: head-only warmup ({HEAD_EPOCHS} epochs, backbone frozen) --")
set_backbone_trainable(model, trainable=False)
best_val_acc, best_state = run_phase("head", HEAD_EPOCHS, head_optimizer, best_val_acc, best_state)

print(f"\n-- Phase 2: full fine-tune ({FINETUNE_EPOCHS} epochs, backbone unfrozen, lr=5e-5) --")
set_backbone_trainable(model, trainable=True)
best_val_acc, best_state = run_phase("finetune", FINETUNE_EPOCHS, finetune_optimizer, best_val_acc, best_state)

print(f"\nBest internal-val video-level accuracy across both phases: {best_val_acc:.4f} "
      f"-- restoring that checkpoint (not necessarily the last epoch) before section 7's "
      f"held-out FF++ evaluation.")
model.load_state_dict(best_state)

print("\nNote: val_video_acc above is measured on a split carved out of TRAIN only (never "
      "the FF++ test set) -- it exists purely so overfitting is visible epoch-by-epoch and "
      "so the checkpoint carried forward is the best one seen during training, not whatever "
      "the last epoch happened to produce. Section 7's FF++ test number below is still the "
      "only number the section 8 gate uses or that should be reported as the held-out result.")


## 7. Evaluate on the held-out FF++ split — frame-level and video-level

Frame-level: does the model classify each individual face crop correctly.
Video-level: mean `P(synthetic)` across a video's valid crops, thresholded at 0.5 —
**this is the number that matters**, and the exact aggregation
`evaluate_cv_transfer.py` reuses against the Nigerian corpus so the two numbers are
comparable.


In [ ]:
def report_metrics(y_true, y_pred, title):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    print(f"\n-- {title} (n={len(y_true)}) --")
    print(f"   accuracy={acc:.4f}  precision={prec:.4f}  recall={rec:.4f}  f1={f1:.4f}")
    print(f"   confusion matrix [rows=true 0/1, cols=pred 0/1]:\n{cm}")
    return {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1,
            "confusion_matrix": cm.tolist(), "n": len(y_true)}


# predict_probs() is defined in section 6, above the training loop -- reused
# here unchanged so this is the exact same forward pass, just against the
# untouched-until-now FF++ test set.
# ---- frame-level ----
frame_probs, frame_labels = predict_probs(test_loader)
frame_preds = (frame_probs >= 0.5).astype(int)
frame_metrics = report_metrics(frame_labels, frame_preds, "FRAME-LEVEL (held-out FF++ test)")

# ---- video-level: mean probability across a video's crops ----
test_crops_with_probs = test_crops.reset_index(drop=True).copy()
test_crops_with_probs["prob_synthetic"] = frame_probs
video_level = test_crops_with_probs.groupby("video_path").agg(
    label=("label", "first"), prob_synthetic=("prob_synthetic", "mean"), n_crops=("label", "size"),
).reset_index()
video_level["pred"] = (video_level["prob_synthetic"] >= 0.5).astype(int)

video_metrics = report_metrics(video_level["label"], video_level["pred"], "VIDEO-LEVEL (held-out FF++ test)")
print(f"\nVideos evaluated: {len(video_level)}/{len(test_videos)} "
      f"(the rest yielded zero face crops -- see the extraction step's warning above)")
print(f"mean crops/video used: {video_level['n_crops'].mean():.1f}")


## 8. THE GATE — do not proceed past a broken pipeline

If `VIDEO_LEVEL_ACCURACY` doesn't clear the threshold below, **this cell raises and
the notebook stops here.** The checkpoint-saving cell after this one will simply
never run. Re-running "Run All" after a failure will stop at this same cell again —
it will not silently save a bad checkpoint on a later attempt.

Why this matters: FF++ under matched (same-domain, same-compression) conditions is
a comparatively easy benchmark in the deepfake-detection literature — published
results with far less careful preprocessing than this notebook's routinely clear
90-99%. A result meaningfully below that on this project's own held-out split means
something in *this* pipeline is broken (bad crops, a labelling bug, a training
bug) — not that deepfake detection is hard in general. Any subsequent number this
model produces on the Nigerian corpus (`evaluate_cv_transfer.py`) would be
meaningless if this gate doesn't pass first; a domain-transfer gap is only
interpretable relative to a credible in-domain baseline.


In [ ]:
CV_TRANSFER_FF_ACCURACY_GATE = 0.90  # mirrors config.CV_TRANSFER_FF_ACCURACY_GATE -- keep both in sync by hand

video_level_accuracy = video_metrics["accuracy"]

if video_level_accuracy < CV_TRANSFER_FF_ACCURACY_GATE:
    raise RuntimeError(
        f"GATE FAILED: video-level FF++ held-out accuracy is {video_level_accuracy:.4f}, "
        f"below the {CV_TRANSFER_FF_ACCURACY_GATE:.2f} gate. Do NOT save this checkpoint or "
        f"trust a Nigerian-corpus number from it. Likely causes to check first: (1) look at "
        f"the sample crops in the sanity-check cell above -- are they actually aligned faces? "
        f"(2) check the train/test label balance printed in section 3 -- is one class starved? "
        f"(3) check training loss actually decreased across epochs in section 6's log. "
        f"Fix the root cause and re-run from the relevant section, don't just lower this gate."
    )

print(f"GATE PASSED: video-level FF++ held-out accuracy {video_level_accuracy:.4f} "
      f">= {CV_TRANSFER_FF_ACCURACY_GATE:.2f}. Proceeding to save the checkpoint.")


## 9. Save checkpoint + FF++ metrics to Drive

Downloaded from Drive into this project's local `models/` directory as:
- `models/cv_watchdog_v1.pt` (matches `config.CV_MODEL_PATH`)
- `models/cv_watchdog_v1_ff_metrics.json` (matches `config.CV_MODEL_FF_METRICS_PATH`)

Checkpoint dict shape mirrors `train_nlp_classifier.py`'s
(`model_state_dict`/`base_model`/`label_map`) so a future `cv_engine`-side loader
can follow the exact same pattern as `nlp_engine.classify_text()`'s.


In [ ]:
# Self-defence: re-check the gate here too, independent of section 8's raise.
# If this cell is ever run on its own (out of "Run All" order) after a failed
# gate, it must still refuse to save -- see the walkthrough note in section 8.
assert video_level_accuracy >= CV_TRANSFER_FF_ACCURACY_GATE, (
    f"Refusing to save: video-level FF++ accuracy {video_level_accuracy:.4f} is below "
    f"the {CV_TRANSFER_FF_ACCURACY_GATE:.2f} gate. This should be unreachable if you ran "
    f"section 8 first and it raised -- if you're seeing this, you jumped straight to this "
    f"cell after a failed gate. Don't save this checkpoint."
)

CHECKPOINT_DIR = Path("/content/drive/MyDrive/watchdog_checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

checkpoint_path = CHECKPOINT_DIR / "cv_watchdog_v1.pt"
metrics_path = CHECKPOINT_DIR / "cv_watchdog_v1_ff_metrics.json"

torch.save({
    "model_state_dict": model.state_dict(),
    "base_model": "efficientnet_b0",
    "input_size": CROP_OUTPUT_SIZE,
    "label_map": {0: "AUTHENTIC", 1: "SYNTHETIC"},
    "imagenet_mean": IMAGENET_MEAN,
    "imagenet_std": IMAGENET_STD,
}, checkpoint_path)

metrics_payload = {
    "trained_on": "FaceForensics++ subset (200 original, 200 Deepfakes, 100 NeuralTextures, c23)",
    "split_method": "StratifiedGroupKFold(n_splits=5), grouped by union-find over reciprocal-pair identity numbers",
    "n_train_videos": len(train_videos), "n_test_videos": len(test_videos),
    "n_fit_videos": len(fit_videos), "n_val_videos": len(val_videos),
    "n_train_crops": len(train_crops), "n_test_crops": len(test_crops),
    "batch_size": BATCH_SIZE,
    "head_epochs": HEAD_EPOCHS, "finetune_epochs": FINETUNE_EPOCHS,
    "augmentation": "RandomResizedCrop(scale=0.85-1.0)+RandomHorizontalFlip+RandomRotation(10)+ColorJitter(0.15)",
    "loss": "unweighted CrossEntropyLoss",
    "best_val_video_accuracy": best_val_acc,
    "frame_level": frame_metrics, "video_level": video_metrics,
    "accuracy_gate": CV_TRANSFER_FF_ACCURACY_GATE,
}
with open(metrics_path, "w") as f:
    json.dump(metrics_payload, f, indent=2)

print(f"Saved checkpoint to: {checkpoint_path}")
print(f"Saved FF++ metrics to: {metrics_path}")
print("\nDownload both into this project's local models/ directory, then run "
      "`python evaluate_cv_transfer.py` locally.")


## Closing notes for the write-up (Chapter 3 methodology, Chapter 4 results)

- **200 videos here are ~100 identity pairs, not 200 independent subjects.** Every
  reciprocal Deepfakes pair (`654_648.mp4`/`648_654.mp4`) shares both source
  identities with its own two originals; treating the 200 as independent samples
  anywhere in the methodology (e.g. a naive per-video train/test split) would be
  the identity-leakage bug this notebook's section 3 specifically prevents.
- **Deepfakes and NeuralTextures share source identities with each other**, not
  just within their own method — the union-find grouping in section 3 spans both
  methods together for exactly this reason. Report the actual number of identity
  groups (printed in section 3), not the raw video count, as the real sample size
  for anything talking about how many distinct people the model was evaluated
  against.
- **A second grouped split (fit/val) is carved out of TRAIN only**, using the same
  union-find identity groups and the same `StratifiedGroupKFold` tool as the
  train/test split, so per-epoch overfitting is visible during training without
  the FF++ test set ever being touched more than the one time section 7 touches
  it. Report this as a two-level split (train/test, then fit/val within train) if
  the methodology chapter shows the split diagram.
- **The gate in section 8 is the credibility check for everything downstream.**
  If `evaluate_cv_transfer.py`'s Nigerian-corpus number is low, that is only a
  meaningful "Western-trained detection degrades on Global South media" finding
  (§2.3.4) *because* this notebook's FF++ number was independently confirmed
  credible first, on the same architecture and the same preprocessing.
- **The one preprocessing asymmetry between this notebook and the Nigerian-corpus
  evaluation: MTCNN runs on CUDA here, CPU in `cv_engine.py`/`evaluate_cv_transfer.py`**
  (see section 4's markdown for the full reasoning) — a necessary consequence of
  this project's CPU-only-outside-Colab constraint, not an oversight, but state it
  as an explicit caveat next to the FF++-vs-Nigerian gap in Chapter 4, not silently.
  Also logged in `docs/PROJECT_CONTEXT.md`.
- **Celeb-DF-v2.zip remains unused.** If this model's Nigerian-corpus transfer
  result is interesting enough to warrant a second evaluation domain, that's a
  follow-up notebook, not a retrofit onto this one.
